# Fase 1 · M06: Grafo de Trazabilidad

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 1 — Transformación |
| **Módulo** | M06 — Grafo Trazabilidad |

---

## 🎯 Qué hace

Genera un grafo interactivo de trazabilidad del pipeline usando networkx, mostrando las relaciones entre tablas intermedias y el dataset final.

## 📋 Requisitos

- `data/01_interim/*.parquet`
- `data/02_processed/df_alumno.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase1/m06_grafo.html` | Grafo interactivo de trazabilidad |

## 🔄 Flujo

```
df_alumno.parquet + interim/*.parquet
    ↓ networkx: construcción del grafo
    ↓ Visualización interactiva
    → docs/html/fase1/m06_grafo.html
```

## ➡️ Siguiente

Fin de Fase 1 — continuar con `f2_m00_indice.ipynb`


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
#
# Importa rutas y constantes de presentación visual.
# COLORES_TABLAS y EMOJIS_TABLAS dan identidad a cada tabla en el grafo.
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
import pandas as pd

# --- Detectar entorno y localizar ROOT ---
def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(
        f"No se encontró carpeta 'src/' subiendo desde {start}. "
        f"Verifica que el notebook está dentro de AU_UJI/"
    )

ROOT = _encontrar_root(Path.cwd())


sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
from src.config import RUTA_INTERIM, RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import formato_numero_es
from src.constantes import TABLA_PREINSCRIPCION, COLORES_TABLAS, EMOJIS_TABLAS
from src.html import generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

# --- Carpeta de salida ---
RUTA_FASE1 = RUTA_HTML / 'fase1'
RUTA_FASE1.mkdir(parents=True, exist_ok=True)

# Atajo para formato español
fmt = formato_numero_es

info_entorno()

✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ============================

In [2]:
# ============================================================================
# CELDA 2: LEER DATOS
# ============================================================================
#
# Carga las métricas de cada tabla (filas, columnas, nombre)
# y del dataset final. Se usa para construir los nodos y tooltips del grafo.
#
# Separa tablas en 2 grupos:
#   - tabs_datos: las 8 del Excel principal (líneas sólidas en el grafo)
#   - tabs_preinsc: preinscripción (línea punteada, origen distinto)
# ============================================================================

print('=' * 60)
print('LEYENDO PARQUETS')
print('=' * 60)

TABLAS = {}
for pq in sorted(RUTA_INTERIM.glob('*.parquet')):
    df = pd.read_parquet(pq)
    TABLAS[pq.stem] = {
        'filas': len(df),
        'columnas': list(df.columns),
        'n_cols': len(df.columns),
        'es_preins': pq.stem == TABLA_PREINSCRIPCION
    }
    print(f'  ✅ {pq.stem:15s}: {fmt(len(df)):>10s} × {len(df.columns)} cols')

# Dataset final
df_path = RUTA_PROCESSED / 'df_alumno.parquet'
if df_path.exists():
    df_alumno = pd.read_parquet(df_path)
    DF_INFO = {'filas': len(df_alumno), 'n_cols': len(df_alumno.columns)}
else:
    DF_INFO = {'filas': 0, 'n_cols': 0}

print(f'\n🎯 df_alumno: {fmt(DF_INFO["filas"])} × {DF_INFO["n_cols"]} cols')

# Separar por origen
tabs_datos = sorted([t for t in TABLAS if not TABLAS[t]['es_preins']])
tabs_preinsc = [t for t in TABLAS if TABLAS[t]['es_preins']]

LEYENDO PARQUETS
  ✅ becas          :     70.524 × 4 cols
  ✅ demograficos   :     30.873 × 5 cols
  ✅ domicilios     :    210.911 × 6 cols
  ✅ expedientes    :    109.568 × 15 cols
  ✅ notas          :    107.908 × 5 cols


  ✅ preinscripcion :    210.986 × 24 cols
  ✅ recibos        :    114.454 × 5 cols
  ✅ titulaciones   :         45 × 6 cols
  ✅ trabajo        :    195.524 × 4 cols

🎯 df_alumno: 109.568 × 37 cols


In [3]:
# ============================================================================
# CELDA 3: GENERAR GRAFO SVG
# ============================================================================
#
# Genera un grafo SVG con la trazabilidad de datos:
#
#   [tabla_1] ──╮
#   [tabla_2] ──┤
#   [tabla_3] ──┤── → [df_alumno]
#   ...         ┤
#   [tabla_8] ──╯
#   [preins]  ╌╌╌╌→ (línea punteada)
#
# Cada nodo muestra: nombre, nº columnas, color de constantes.
# Las líneas son curvas Bézier (path SVG con control points).
# Los tooltips muestran filas + lista de columnas (máx 5).
# ============================================================================

print('=' * 60)
print('GENERANDO GRAFO SVG')
print('=' * 60)

fmt = formato_numero_es

# Configuración del SVG
svg_width = 900
svg_height = 600
node_width = 120
node_height = 50
target_x = 750
target_y = 275
target_width = 130
target_height = 60

# Generar nodos SVG para tablas datos_proyecto
nodes_svg = ''
lines_svg = ''
y_start = 30
y_spacing = 65

for i, t in enumerate(tabs_datos):
    info = TABLAS[t]
    color = COLORES_TABLAS.get(t, '#3182ce')
    x = 50
    y = y_start + i * y_spacing
    
    # Línea curva al destino
    cx1 = x + node_width + 100
    cy1 = y + node_height/2
    cx2 = target_x - 100
    cy2 = target_y + target_height/2
    lines_svg += f'''<path d="M {x + node_width} {y + node_height/2} C {cx1} {cy1}, {cx2} {cy2}, {target_x} {target_y + target_height/2}" 
        stroke="{color}" stroke-width="2" fill="none" opacity="0.7"/>
    '''
    
    # Tooltip con columnas
    cols_text = ', '.join(info['columnas'][:5])
    if len(info['columnas']) > 5:
        cols_text += f'... (+{len(info["columnas"])-5})'
    
    # Nodo
    nodes_svg += f'''
    <g class="node" data-tabla="{t}">
        <rect x="{x}" y="{y}" width="{node_width}" height="{node_height}" rx="8" 
              fill="{color}" stroke="#2d3748" stroke-width="1.5"/>
        <text x="{x + node_width/2}" y="{y + 20}" text-anchor="middle" 
              fill="white" font-weight="bold" font-size="12">{t}</text>
        <text x="{x + node_width/2}" y="{y + 36}" text-anchor="middle" 
              fill="rgba(255,255,255,0.8)" font-size="10">{info['n_cols']} cols</text>
        <title>{t}: {fmt(info['filas'])} filas, {info['n_cols']} cols&#10;{cols_text}</title>
    </g>
    '''

# Nodo preinscripción (con línea punteada)
for t in tabs_preinsc:
    info = TABLAS[t]
    color = '#805ad5'
    x = 300
    y = svg_height - 100
    
    # Línea punteada al destino
    lines_svg += f'''<path d="M {x + node_width} {y + node_height/2} C {x + 200} {y}, {target_x - 50} {target_y + target_height + 50}, {target_x + target_width/2} {target_y + target_height}" 
        stroke="#ed8936" stroke-width="2" fill="none" stroke-dasharray="8,4" opacity="0.8"/>
    '''
    
    cols_text = ', '.join(info['columnas'][:5]) + '...'
    
    nodes_svg += f'''
    <g class="node" data-tabla="{t}">
        <rect x="{x}" y="{y}" width="{node_width + 20}" height="{node_height}" rx="8" 
              fill="{color}" stroke="#2d3748" stroke-width="1.5"/>
        <text x="{x + (node_width+20)/2}" y="{y + 20}" text-anchor="middle" 
              fill="white" font-weight="bold" font-size="12">{t}</text>
        <text x="{x + (node_width+20)/2}" y="{y + 36}" text-anchor="middle" 
              fill="rgba(255,255,255,0.8)" font-size="10">{info['n_cols']} cols</text>
        <title>{t}: {fmt(info['filas'])} filas, {info['n_cols']} cols</title>
    </g>
    '''

# Nodo destino (df_alumno)
nodes_svg += f'''
<g class="node target">
    <rect x="{target_x}" y="{target_y}" width="{target_width}" height="{target_height}" rx="10" 
          fill="#38a169" stroke="#276749" stroke-width="2"/>
    <text x="{target_x + target_width/2}" y="{target_y + 25}" text-anchor="middle" 
          fill="white" font-weight="bold" font-size="14">df_alumno</text>
    <text x="{target_x + target_width/2}" y="{target_y + 43}" text-anchor="middle" 
          fill="rgba(255,255,255,0.85)" font-size="11">{DF_INFO['n_cols']} cols</text>
    <title>df_alumno: {fmt(DF_INFO['filas'])} filas, {DF_INFO['n_cols']} columnas</title>
</g>
'''

# SVG completo
grafo_svg = f'''
<svg viewBox="0 0 {svg_width} {svg_height}" style="width:100%;height:auto;max-height:600px;background:#f8fafc;border-radius:10px;">
    <defs>
        <marker id="arrowhead" markerWidth="10" markerHeight="7" refX="9" refY="3.5" orient="auto">
            <polygon points="0 0, 10 3.5, 0 7" fill="#718096"/>
        </marker>
    </defs>
    
    <!-- Líneas -->
    {lines_svg}
    
    <!-- Nodos -->
    {nodes_svg}
</svg>
'''

print(f'  Nodos: {len(tabs_datos) + len(tabs_preinsc) + 1}')
print(f'  Conexiones: {len(tabs_datos) + len(tabs_preinsc)}')

GENERANDO GRAFO SVG
  Nodos: 10
  Conexiones: 9


In [4]:
# ============================================================================
# CELDA 4: GENERAR PÁGINA HTML
# ============================================================================
#
# Construye m06_grafo.html con:
#   - Leyenda de colores (Excel principal, Preinscripción, df_alumno)
#   - Grafo SVG generado en celda 3
#   - Panel lateral con resumen de cada tabla (filas × columnas)
#   - CSS responsivo (panel lateral pasa abajo en pantallas < 900px)
#
# Los estilos CSS se pasan como estilos_adicionales a render_base_html()
# para que se inyecten en el <head> del HTML.
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(fase_activa='fase1', modulo_activo='m06')

# Panel de información
info_panel = '''
<div class="grafo-info-panel">
    <h4>📊 Resumen de Trazabilidad</h4>
    <div class="grafo-stats">
'''

for t in tabs_datos:
    info = TABLAS[t]
    color = COLORES_TABLAS.get(t, '#3182ce')
    emoji = EMOJIS_TABLAS.get(t, '📊')
    info_panel += f'''
        <div class="grafo-stat-item" style="border-left-color:{color}">
            <span class="stat-emoji">{emoji}</span>
            <div class="stat-info">
                <div class="stat-name">{t}</div>
                <div class="stat-detail">{fmt(info['filas'])} × {info['n_cols']}</div>
            </div>
        </div>
    '''

for t in tabs_preinsc:
    info = TABLAS[t]
    info_panel += f'''
        <div class="grafo-stat-item" style="border-left-color:#805ad5">
            <span class="stat-emoji">📄</span>
            <div class="stat-info">
                <div class="stat-name">{t}</div>
                <div class="stat-detail">{fmt(info['filas'])} × {info['n_cols']}</div>
            </div>
        </div>
    '''

info_panel += f'''
        <div class="grafo-stat-item grafo-stat-result" style="border-left-color:#38a169">
            <span class="stat-emoji">🎯</span>
            <div class="stat-info">
                <div class="stat-name">df_alumno</div>
                <div class="stat-detail">{fmt(DF_INFO['filas'])} × {DF_INFO['n_cols']}</div>
            </div>
        </div>
    </div>
</div>
'''

# Contenido completo
contenido = f'''
<div class="grafo-leyenda">
    <div class="leyenda-item"><div class="leyenda-color" style="background:#3182ce"></div>datos_proyecto ({len(tabs_datos)})</div>
    <div class="leyenda-item"><div class="leyenda-color" style="background:#805ad5"></div>preinscripcion ({len(tabs_preinsc)})</div>
    <div class="leyenda-item"><div class="leyenda-color" style="background:#38a169"></div>df_alumno</div>
</div>

<div style="text-align:right; margin:10px 0;">
    <a href="m06_grafo_pyvis.html" style="display:inline-block; padding:10px 20px; background:linear-gradient(135deg, #805ad5, #6b46c1); color:white; border-radius:8px; text-decoration:none; font-size:0.9em; font-weight:bold; box-shadow:0 2px 8px rgba(128,90,213,0.3); transition:transform 0.2s;" onmouseover="this.style.transform='scale(1.05)'" onmouseout="this.style.transform='scale(1)'">🕸️ Abrir grafo interactivo (PyVis)</a>
</div>

<div class="grafo-container">
    <div class="grafo-svg-wrapper">
        {grafo_svg}
    </div>
    {info_panel}
</div>
'''

# CSS adicional para el grafo
estilos_adicionales = '''
<style>
.grafo-container {
    display: grid;
    grid-template-columns: 1fr 280px;
    gap: 20px;
    margin-top: 15px;
}

.grafo-svg-wrapper {
    background: white;
    border-radius: 12px;
    padding: 20px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
}

.grafo-svg-wrapper svg .node {
    cursor: pointer;
    transition: opacity 0.2s;
}

.grafo-svg-wrapper svg .node:hover {
    opacity: 0.85;
}

.grafo-svg-wrapper svg .node rect {
    filter: drop-shadow(2px 2px 4px rgba(0,0,0,0.15));
}

.grafo-info-panel {
    background: white;
    border-radius: 12px;
    padding: 18px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
}

.grafo-info-panel h4 {
    color: #2d3748;
    margin-bottom: 15px;
    padding-bottom: 10px;
    border-bottom: 2px solid #e2e8f0;
}

.grafo-stats {
    display: flex;
    flex-direction: column;
    gap: 8px;
}

.grafo-stat-item {
    display: flex;
    align-items: center;
    gap: 10px;
    padding: 8px 10px;
    background: #f7fafc;
    border-radius: 8px;
    border-left: 4px solid #3182ce;
}

.grafo-stat-result {
    background: #f0fff4;
    margin-top: 10px;
}

.stat-emoji {
    font-size: 1.2em;
}

.stat-info {
    flex: 1;
}

.stat-name {
    font-weight: bold;
    font-size: 0.85em;
    color: #2d3748;
}

.stat-detail {
    font-size: 0.75em;
    color: #718096;
}

@media (max-width: 900px) {
    .grafo-container {
        grid-template-columns: 1fr;
    }
}
</style>
'''

# Render HTML
html = render_base_html(
    titulo='M06: Grafo',
    icono='🕸️',
    subtitulo='Fase 1: Trazabilidad de Datos',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=contenido,
    estilos_adicionales=estilos_adicionales,
    notebook_nombre='f1_m06_grafo.ipynb',
    notebook_carpeta='fase1_transformacion'
)

guardar_html(html, RUTA_FASE1 / 'm06_grafo.html')
print('\n✅ m06_grafo.html')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m06_grafo.html

✅ m06_grafo.html
